# Colab runner

Runs the project on a Colab GPU from VS Code.

**Every new session:** run cells 1 and 2 first, in order.

- Code comes from GitHub (always the latest pushed version).
- Big files (`data/`, `outputs/`, `judges/`) live on Google Drive, so they survive when the session ends.

## 1. Connect Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Get the latest code and install it

Paste your GitHub repository address between the quotes on the first line (once).

In [2]:
REPO = "https://github.com/tca-codes/moral-rag-cf.git"   # e.g. "https://github.com/your-username/moral-rag-cf.git"
DRIVE_DIR = "/content/drive/MyDrive/moral-rag-cf"

import os
assert REPO, "Paste your GitHub repository address into REPO on the first line of this cell."
assert os.path.isdir("/content/drive/MyDrive"), "Google Drive is not connected. Run cell 1 first."

if not os.path.isdir("/content/moral-rag-cf/.git"):
    !git clone -q {REPO} /content/moral-rag-cf
%cd /content/moral-rag-cf
!git pull -q
!pip install -q -r requirements.txt
!pip install -q -e .

# data, outputs and judges are stored on Drive and linked into the code folder
for name in ["data", "outputs", "judges"]:
    os.makedirs(f"{DRIVE_DIR}/{name}", exist_ok=True)
    if not os.path.exists(name):
        os.symlink(f"{DRIVE_DIR}/{name}", name)

print("\nCode version in use:")
!git log -1 --format="%h  %s  (%cr)"
!nvidia-smi --query-gpu=name,memory.total --format=csv

/content/moral-rag-cf
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for mvrag (pyproject.toml) ... done

Code version in use:
72a71b3  Created using Colab  (15 minutes ago)
name, memory.total [MiB]
NVIDIA L4, 23034 MiB


## 3. Speed test (40 questions, 7B model, uncompressed)

Use this on an L4 or A100. On a T4, remove `--no-4bit` and change 16 to 2.

In [3]:
!python scripts/run_agents.py --split test --n 40 --max-new-tokens 60 --model Qwen/Qwen2.5-7B-Instruct --no-4bit --questions-per-chunk 16 --out outputs/speedtest.jsonl

40 questions selected from split=test
config.json: 100% 663/663 [00:00<00:00, 3.25MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 23.7MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 114MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 116MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 159MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 100% 27.8k/27.8k [00:00<00:00, 69.0MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/3.56G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/7.42G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/11.4G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   2% 256M/15.2G [00:04<03:04, 81.2MB/s, 45.0MB/s  ]
Reconstructing (incomplete total...):  15% 2.29G/15.2G [00:11<01:07, 191MB/s,  115MB/s  ]
Reconstructing (incomplete to

In [5]:
%cd /content/moral-rag-cf
!cp --update=none outputs/speedtest.jsonl outputs/agents_test_7B_full.jsonl
!python scripts/run_agents.py --split test --n 4000 --max-new-tokens 60 --model Qwen/Qwen2.5-7B-Instruct --no-4bit --questions-per-chunk 16 --out outputs/agents_test_7B_full.jsonl
!python scripts/run_agents.py --split test --n 4000 --max-new-tokens 60 --model Qwen/Qwen2.5-7B-Instruct --no-4bit --questions-per-chunk 16 --plain --out outputs/plain_test_7B_full.jsonl
!python scripts/table1.py --plain outputs/plain_test_7B_full.jsonl --agents outputs/agents_test_7B_full.jsonl --out outputs/table1_7B_full.json
!python scripts/score_agents.py --agents outputs/agents_test_7B_full.jsonl --out outputs/judge_scores_7B_full.json

/content/moral-rag-cf
3753 questions selected from split=test
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:04<00:00, 76.17it/s]
Done. 0 new questions written to outputs/agents_test_7B_full.jsonl
3753 questions selected from split=test
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:04<00:00, 76.59it/s]
Done. 0 new questions written to outputs/plain_test_7B_full.jsonl
Loading weights: 100% 105/105 [00:00<00:00, 3364.88it/s]
Loading weights: 100% 105/105 [00:00<00:00, 3398.24it/s]
Loading weights: 100% 105/105 [00:00<00:00, 3577.19it/s]
Loading weights: 100% 105/105 [00:00<00:00, 3959.35it/s]
Loading weights: 100% 105/105 [00:00<00:00, 3016.82it/s]
questions: 3753

moral value    plain model  moral agent   change
care               79.53%       87.18%   +7.65%
fairness           85.71%       89.18%   +3.48%
loyalty            88.40%       93.50%   +5.10%
authority          78.72%     

## When you are done

Disconnect the runtime so it stops using compute units: open the Command Palette and run **Colab: Remove Server**.